

---

# Практична робота №3 — Tidy Data: приведення до охайного формату, шкали вимірювання та збереження даних (CSV / JSON)

**Студент:** Антонов Андрій

**Група:** IT-42

**Обраний набір для розрахунків:** **варіант №1 — Київ**

**Мета:** перетворити таблицю з wide-формату в tidy (long) і назад, класифікувати змінні за шкалою вимірювання та прочитати й записати таблицю в форматах CSV і JSON.

---

## Завдання 1. Побудова wide-таблиці свого варіанта

Створюємо початковий DataFrame для Києва у `wide`-форматі, де кожен місяць представлений окремим стовпцем.

In [ ]:
import pandas as pd

wide_df = pd.DataFrame({
    "Місто": ["Київ"],
    "Січ": [78],
    "Лют": [74],
    "Бер": [68],
    "Кві": [58],
    "Тра": [50],
    "Чер": [44],
    "Лип": [40],
    "Сер": [42],
    "Вер": [52],
    "Жов": [66],
    "Лис": [76],
    "Гру": [82]
})

print("Wide-таблиця:")
print(wide_df)
print("\nФорма (shape):", wide_df.shape)

**Результат виконання:**

```text
Wide-таблиця:
  Місто  Січ  Лют  Бер  Кві  Tра  Чер  Лип  Сер  Вер  Жов  Лис  Гру
0  Київ   78   74   68   58   50   44   40   42   52   66   76   82

Форма (shape): (1, 13)

```

---

## Завдання 2. Приведення до tidy (melt)

Використовуємо метод `.melt()`, щоб перетворити таблицю з широкого формату в охайний (`tidy`/`long`). Назва стовпця з містом залишається незмінною (`id_vars`), а місяці та показники хмарності переходять у два нові стовпці.

In [ ]:
tidy_df = wide_df.melt(
    id_vars="Місто",
    var_name="місяць",
    value_name="хмарність"
)

print("Tidy-таблиця (перші 6 рядків):")
print(tidy_df.head(6))
print("\nФорма (shape) tidy-таблиці:", tidy_df.shape)

**Результат виконання:**

```text
Tidy-таблиця (перші 6 рядків):
   Місто місяць  хмарність
0  Київ    Січ         78
1  Київ    Лют         74
2  Київ    Бер         68
3  Київ    Кві         58
4  Київ    Тра         50
5  Київ    Чер         44

Форма (shape) tidy-таблиці: (12, 3)

```

**Пояснення:** У вихідній `wide`-таблиці був лише 1 рядок і 13 стовпців (1 для міста + 12 для місяців). Після застосування `melt()` таблиця має форму `(12, 3)`, тому що кожен із 12 колишніх стовпців-місяців розгорнувся в окремий рядок у стовпці `місяць`, утворивши 12 рядків і 3 основні колонки (`Місто`, `місяць`, `хмарність`).

---

## Завдання 3. Класифікація змінних

Визначення шкали вимірювання для кожного зі стовпців `tidy`-таблиці:

1. **`Місто` — Номінальна шкала (nominal).**
*Пояснення:* Назви міст є категоріями без природного порядку чи ранжування (неможливо сказати, що «Київ» більше чи менше за «Львів»).
2. **`місяць` — Порядкова шкала (ordinal).**
*Пояснення:* Назви місяців утворюють чітку хронологічну послідовність (січень іде до лютого, лютий до березня тощо), хоча математична відстань між ними не задана через їхню назву.
3. **`хмарність` — Шкала відношень (ratio).**
*Пояснення:* Значення відсотків хмарності мають абсолютний, незмінюваний нуль (0% означає повну відсутність хмарності, а відношення значень мають фізичний сенс — наприклад, 80% є вдвічі більшою хмарністю, ніж 40%).

---

## Завдання 4. Зворотне перетворення й перевірка

Повертаємося до `wide`-формату за допомогою методу `.pivot()`, щоб переконатися у відсутності втрати даних.

In [ ]:
back_wide = tidy_df.pivot(
    index="Місто",
    columns="місяць",
    values="хмарність"
).reset_index()

print("Відновлена wide-таблиця:")
print(back_wide)

**Результат виконання:**

```text
Відновлена wide-таблиця:
місяць Місто  Бер  Гру  Жов  Кві  Лип  Лис  Лют  Сер  Січ  Тра  Чер  Вер
0       Київ   68   82   66   58   40   76   74   42   78   50   44   52

```

**Порівняння:** Відновлена таблиця повністю збігається за числовими значеннями з початковою таблицею з Завдання 1 (порядок стовпців-місяців після `pivot` може автоматично відсортуватися за алфавітом, що є нормальною поведінкою).

---

## Завдання 5. Читання й запис (CSV та JSON)

Зберігаємо `tidy`-таблицю у файли формату CSV та JSON, після чого завантажуємо їх назад і перевіряємо цілісність даних.

In [ ]:
# 1. Запис файлів
tidy_df.to_csv("variant.csv", index=False)
tidy_df.to_json("variant.json", orient="records", force_ascii=False)

# 2. Читання назад у нові DataFrame
reloaded_csv = pd.read_csv("variant.csv")
reloaded_json = pd.read_json("variant.json")

# 3. Перевірка
print("Чи збігається завантажений CSV з оригіналом:", tidy_df.equals(reloaded_csv))
print("Чи збігається завантажений JSON з оригіналом:", tidy_df.equals(reloaded_json))

**Результат виконання:**

```text
Чи збігається завантажений CSV з оригіналом: True
Чи збігається завантажений JSON з оригіналом: True

```

### Порівняння структур файлів CSV та JSON:

* **CSV (Comma-Separated Values):** Це текстовий формат таблиці, де дані розділені комами (або іншими розділюваниками), а рядки розташовані на окремих рядках. Перший рядок зазвичай містить заголовки стовпців. Він максимально простий, компактний і читабельний для людини, але не має вбудованого опису типів даних.
* **JSON (JavaScript Object Notation):** Це структура у вигляді списку об'єктів (або словників) із ключами та значеннями (у нашому випадку з параметром `orient="records"`). Файл містить синтаксичні елементи (фігурні дужки `{ }`, квадратні дужки `[ ]`, лапки), завдяки чому чітко визначає ієрархію та структуру записів, ідеально підходячи для передачі даних через веб-API.

---

## Письмові відповіді на контрольні запитання

1. **Чи можете пояснити, чому стовпець "місто" — номінальна шкала, а стовпець "хмарність" — шкала відношень?**
Стовпець «місто» представляє назви категорій без жодного кількісного чи порядкового значення (номінальна шкала). Стовпець «хмарність» вимірюється у відсотках від 0 до 100, де існує абсолютний нуль (абсолютна відсутність хмарності), а значеннєві співвідношення коректні (наприклад, 60% хмарності вдвічі більші за 30%), що повністю відповідає шкалі відношень.
2. **Чи розумієте, чому вихідна wide-таблиця з місяцями в стовпцях не вважається tidy, і що саме змінюється в її структурі після melt()?**
Вихідна `wide`-таблиця не є `tidy`, оскільки назви стовпців (місяці) насправді є **значеннями змінної часу**, а не назвами самих змінних. Метод `.melt()` перетворює заголовки стовпців на значення в окремому рядковому стовпці (`місяць`), приводячи таблицю до стандарту, де кожен стовпець — це змінна, а кожен рядок — одне спостереження.
3. **Чи можете описати, чим формат CSV принципово відрізняється від JSON при зберіганні однієї й тієї самої таблиці?**
CSV зберігає таблицю у вигляді плоского текстового рядкового потоку, де стовпці розмежовані комами. JSON організовує дані у вигляді ієрархічних структур (масивів об'єктів із парами ключ-значення), що робить його більш гнучким для складних вкладених структур даних, хоча він вимагає більше символів для службового синтаксису (дужок, лапок).

---

## Висновок

Практичну роботу успішно виконано. Таблицю хмарності для Києва (`wide`-формат) переведено в охайний `tidy`-формат за допомогою методу `.melt()`. Проведено класифікацію змінних за шкалами вимірювання (номінальна, порядкова, відношень). За допомогою `.pivot()` виконано зворотне перетворення та перевірено ідентичність даних. Здійснено збереження та читання датасету у форматах CSV і JSON із підтвердженням цілісності інформації.